In [1]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from qdrant_client.http.models import PointStruct
from qdrant_client.http.models import Filter, FieldCondition, MatchValue

from pprint import pprint

In [2]:
client = QdrantClient(host="localhost", port=6333)
## In-memory for quick tests
#client = QdrantClient(":memory:")
## No concurrent access allowed for this path
#client = QdrantClient(path="/tmp/path_qdrant_db")

In [3]:
COLLECTION_NAME = "test_collection"
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=4, distance=Distance.DOT),
)


True

In [4]:
operation_info = client.upsert(
    collection_name=COLLECTION_NAME,
    wait=True,
    points=[
        PointStruct(id=1,vector=[0.82, 0.78, 0.12, 0.10],payload={"continente": "Europa", "cidade": "Berlim"}),
        PointStruct(id=2,vector=[0.88, 0.74, 0.09, 0.14],payload={"continente": "Europa", "cidade": "Paris"}),
        PointStruct(id=3,vector=[0.79, 0.83, 0.11, 0.08],payload={"continente": "Europa", "cidade": "Londres"}),
        PointStruct(id=40,vector=[0.15, 0.18, 0.91, 0.11],payload={"continente": "America", "cidade": "Nova York"}),
        PointStruct(id=50,vector=[0.19, 0.11, 0.87, 0.15],payload={"continente": "America", "cidade": "Toronto"}),
        PointStruct(id=60,vector=[0.12, 0.21, 0.84, 0.10],payload={"continente": "America", "cidade": "Sao Paulo"}),
        PointStruct(id=700,vector=[0.11, 0.09, 0.18, 0.88],payload={"continente": "Asia", "cidade": "Pequim"}),
        PointStruct(id=800,vector=[0.08, 0.12, 0.14, 0.91],payload={"continente": "Asia", "cidade": "Toquio"}),
        PointStruct(id=900,vector=[0.14, 0.07, 0.21, 0.83],payload={"continente": "Asia", "cidade": "Mumbai"}),
    ],
)

In [ ]:
results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=[0.85, 0.80, 0.10, 0.10],
    limit=3,
)

# pprint(results)
for i, point in enumerate(results.points, start=1):
    cidade = point.payload["cidade"]
    continente = point.payload["continente"]
    score = round(point.score, 4)

    print(f"{i}. {cidade} ({continente}) - similaridade: {score}")


1. Paris (Europa) - similaridade: 1.363
2. Londres (Europa) - similaridade: 1.3545
3. Berlim (Europa) - similaridade: 1.343


In [ ]:
results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=[0.85, 0.80, 0.10, 0.10],
    query_filter=Filter(
        must=[
            FieldCondition(
                key="continente",
                match=MatchValue(value="Asia"),
            )
        ]
    ),
    limit=3,
)

print("Resultados filtrados:\n")

for i, point in enumerate(results.points, start=1):
    cidade = point.payload["cidade"]
    continente = point.payload["continente"]
    score = point.score

    print(
        f"{i}. Cidade: {cidade}"
        f" | Continente: {continente}"
        f" | Similaridade: {score:.4f}"
    )

Resultados filtrados:

1. Cidade: Mumbai | Continente: Asia | Similaridade: 0.2790
2. Cidade: Pequim | Continente: Asia | Similaridade: 0.2715
3. Cidade: Toquio | Continente: Asia | Similaridade: 0.2690


In [ ]:
client.upsert(
    collection_name=COLLECTION_NAME,
    wait=True,
    points=[
        PointStruct(id=1000,
          vector=[0.85, 0.80, 0.10, 0.10],
          payload={"continente": "Nulo", "cidade": "QUERY" })
        ],
)
 

UpdateResult(operation_id=6, status=<UpdateStatus.COMPLETED: 'completed'>)